# Spotify Top 10 Songs (2010-2019): Predicting Song Popularity

**Goal:** classify a song as High or Low popularity from its audio features and genre.

**Dataset:** `top10s.csv` (603 songs, Spotify top songs by year).

**Method note:** the target is built from `pop`, so `pop` is excluded from the features to avoid target leakage. A baseline and cross-validation are included so the results can be trusted.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

sns.set(style="whitegrid")
%matplotlib inline

## 1. Load and inspect

In [ ]:
df = pd.read_csv("top10s.csv", encoding="latin1")
print(df.shape)
df.head()

In [ ]:
df.info()
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
df.describe().T

## 2. Cleaning
Drop the index column, rename columns, and check rows with impossible zeros (bpm, energy, danceability or popularity equal to 0).

In [ ]:
df = df.drop(columns=["Unnamed: 0"]).rename(columns={"top genre": "top_genre", "dB": "db"})

zero_rows = df[(df.bpm == 0) | (df.nrgy == 0) | (df.dnce == 0) | (df["pop"] == 0)]
print("Rows with a zero in bpm/nrgy/dnce/pop:", len(zero_rows))
zero_rows[["title", "artist", "bpm", "nrgy", "dnce", "pop"]]

In [ ]:
# Only 5 rows, clearly bad data, so drop them
df = df.drop(zero_rows.index).reset_index(drop=True)
print("Rows after cleaning:", len(df))

## 3. Exploratory analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["pop"], bins=20, kde=True, ax=ax[0]); ax[0].set_title("Popularity distribution")
df.groupby("year")["pop"].mean().plot(marker="o", ax=ax[1]); ax[1].set_title("Average popularity by year")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
df["top_genre"].value_counts().head(10).plot(kind="bar", ax=ax[0])
ax[0].set_title("Top 10 genres"); ax[0].tick_params(axis="x", rotation=45)
df["artist"].value_counts().head(10).plot(kind="bar", ax=ax[1], color="tab:orange")
ax[1].set_title("Top 10 artists by number of chart songs"); ax[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Average popularity of the most common genres
top_genres = df["top_genre"].value_counts().head(8).index
df[df["top_genre"].isin(top_genres)].groupby("top_genre")["pop"].mean().sort_values().plot(kind="barh", figsize=(8, 4))
plt.title("Average popularity by genre (8 most common)")
plt.xlabel("Average popularity")
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation heatmap")
plt.show()

**Takeaway:** `pop` has weak correlation with every audio feature (the strongest is `year`, about 0.24). So a model should not be able to predict popularity very well. If one scores near 100%, something is wrong.

## 4. Target and features
Target: High if `pop` is at or above the median, else Low.

**Important:** `pop` is removed from the features, because the target is built from it.

In [ ]:
median_pop = df["pop"].median()
df["popularity_level"] = np.where(df["pop"] >= median_pop, "High", "Low")
print("Median popularity:", median_pop)
print(df["popularity_level"].value_counts())

df_encoded = pd.get_dummies(df, columns=["top_genre"], drop_first=True)

# 'pop' is excluded here to avoid target leakage
feature_cols = [c for c in df_encoded.columns
                if c not in ["title", "artist", "popularity_level", "pop"]]
assert "pop" not in feature_cols

X = df_encoded[feature_cols]
y = df_encoded["popularity_level"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

## 5. Models
A baseline (always predict the majority class) is included so the models can be judged against something. Scaling is inside a pipeline so it is fit only on training data.

In [ ]:
models = {
    "Baseline (majority class)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows, preds = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds[name] = model.predict(X_test)
    cv_scores = cross_val_score(model, X, y, cv=cv)
    rows.append({"Model": name,
                 "Test accuracy": round(accuracy_score(y_test, preds[name]), 3),
                 "5-fold CV mean": round(cv_scores.mean(), 3),
                 "CV std": round(cv_scores.std(), 3)})

results = pd.DataFrame(rows).set_index("Model")
results

In [ ]:
best = "Logistic Regression"
print(classification_report(y_test, preds[best]))

cm = confusion_matrix(y_test, preds[best], labels=["Low", "High"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Low", "High"], yticklabels=["Low", "High"])
plt.title(f"{best}: confusion matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.show()

## 6. Feature importance (Random Forest)

In [ ]:
rf = models["Random Forest"]
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.head(10).plot(kind="barh", figsize=(8, 5)).invert_yaxis()
plt.title("Top 10 features (Random Forest)")
plt.show()

## 7. Conclusion
- `pop` is excluded from the features because the target is derived from it (including it would leak the answer and give a meaningless near-100% score).
- The majority-class baseline is about 53%. **Logistic Regression performed best** (about 66% test accuracy, about 63% in 5-fold CV), roughly 10 to 13 points above the baseline.
- Decision Tree and Random Forest did worse than Logistic Regression, likely because the dataset is small (about 600 songs) and the signal is weak.
- No single feature dominates. Duration, danceability, valence, tempo and energy have similar importance, so audio features alone say little about popularity.
- **Limitations:** small dataset, popularity split at the median, and popularity depends on things outside the audio features (artist fame, marketing, release timing).
- **Possible next steps:** tune the models, try artist-level features, or predict popularity as a number (regression).